# Marketing Channel Effectiveness and Sales Prediction

## 1. Business Understanding

### Business Problem

A company invests in multiple advertising channels, including TV, Radio,
and Newspaper, to promote its products.

The company wants to understand how advertising expenditure across
these channels relates to product sales.

The objective is to analyze historical advertising data, identify
relationships between advertising expenditure and sales, and develop
statistical models to predict sales.

### Project Objectives

1. Understand the structure and characteristics of the dataset.
2. Explore relationships between advertising channels and sales.
3. Develop Simple and Multiple Linear Regression models.
4. Evaluate the statistical significance of model coefficients.
5. Assess model assumptions and diagnostic measures.
6. Compare model performance and develop business insights.

### Business Questions

- How does advertising expenditure across different channels relate to sales?
- Which advertising channel has the strongest relationship with sales?
- Can advertising expenditure be used to predict product sales?
- Does combining advertising channels improve predictive performance?
- Can the analysis provide useful insights for advertising budget planning?

### Dataset Description

The project uses the Advertising dataset from the
Introduction to Statistical Learning (ISLR) resource.

The dataset contains advertising expenditure across three channels
and the corresponding product sales observations.

**Source:** https://www.statlearning.com/resources-first-edition

### Important Consideration

The dataset is observational. Therefore, regression coefficients
and relationships identified in this analysis should not automatically
be interpreted as causal advertising effects.

## 2. Importing Libraries

The project uses Pandas and NumPy for data manipulation and numerical
operations, along with Matplotlib and Seaborn for visualization.

In [1]:
import sys
from pathlib import Path

# Get the project root directory
PROJECT_ROOT = Path.cwd().parent

# Add project root to Python's module search path
sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: c:\Users\Disha\OneDrive\Desktop\Marketing Spend vs Revenue Analysis


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils import load_and_split

df_train, df_test = load_and_split(
    path=PROJECT_ROOT / "Dataset" / "Advertising.csv"
)

print("Training shape:", df_train.shape)
print("Testing shape:", df_test.shape)

Training shape: (160, 4)
Testing shape: (40, 4)


## 3. Data Loading

The dataset is loaded using the reusable function defined in
`src/utils.py`.

The dataset is divided into training and testing subsets using an
80:20 split.

The training dataset will be used for data understanding, EDA,
model development and model selection.

The testing dataset will remain untouched until the final model
evaluation stage.

In [3]:
df_train, df_test = load_and_split()

# Use training data for data understanding
df = df_train.copy()

print("Training dataset shape:", df_train.shape)
print("Testing dataset shape:", df_test.shape)

Training dataset shape: (160, 4)
Testing dataset shape: (40, 4)


## 4. Dataset Overview

The following steps examine the first and last few observations
to understand how the data is organized.

In [4]:
df.head() # displays the 1st 5 rows of the dataset

,tv,radio,newspaper,sales
80,116.0,7.7,23.1,11.0
198,177.0,9.3,6.4,12.8
39,43.1,26.7,35.1,10.1
25,62.3,12.6,18.3,9.7
123,224.0,2.4,15.6,11.6


In [5]:
df.tail() # displays the last 5 rows of the dataset

,tv,radio,newspaper,sales
107,25.0,11.0,29.7,7.2
15,204.1,32.9,46.0,19.0
93,217.7,33.5,59.0,19.4
180,165.6,10.0,17.6,12.6
103,280.2,10.1,21.4,14.8


In [6]:
print("Number of observations:", df.shape[0])
print("Number of features:", df.shape[1])

Number of observations: 160
Number of features: 4


In [7]:
print("Dataset columns:")
print(df.columns.tolist())

Dataset columns:
['tv', 'radio', 'newspaper', 'sales']


In [8]:
df.info() # gives the summary -- data types, non-null values, memory usage

<class 'pandas.core.frame.DataFrame'>
Index: 160 entries, 80 to 103
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   tv         160 non-null    float64
 1   radio      160 non-null    float64
 2   newspaper  160 non-null    float64
 3   sales      160 non-null    float64
dtypes: float64(4)
memory usage: 6.2 KB


## 5. Descriptive Statistics

Descriptive statistics provide an overview of the numerical
characteristics of each variable.

The measures include:
- Mean
- Standard deviation
- Minimum and maximum
- Quartiles

In [9]:
df.describe() # gives statistical summary

,tv,radio,newspaper,sales
count,160.000000,160.000000,160.000000,160.000000
mean,150.019375,22.875625,29.945625,14.100000
std,84.418857,14.805216,20.336449,5.108754
min,0.700000,0.000000,0.300000,1.600000
25%,77.750000,9.825000,12.875000,10.475000
50%,150.650000,21.200000,25.600000,13.200000
75%,218.825000,36.425000,44.500000,17.325000
max,296.400000,49.600000,100.900000,27.000000


## 6. Data Quality Assessment

Before proceeding with exploratory data analysis, the dataset
is checked for:

1. Missing values.
2. Duplicate observations.
3. Unexpected negative values.
4. Incorrect data types.

In [10]:
print("Missing values in each column:")
print(df.isnull().sum())

print("\nTotal missing values:", df.isnull().sum().sum())

Missing values in each column:
tv           0
radio        0
newspaper    0
sales        0
dtype: int64

Total missing values: 0


In [11]:
print("Number of duplicate rows:", df.duplicated().sum())

Number of duplicate rows: 0


In [12]:
print("Negative values in each column:")
print((df < 0).sum())
assert (df >= 0).all().all(), "Negative values detected in the dataset!"

Negative values in each column:
tv           0
radio        0
newspaper    0
sales        0
dtype: int64


In [13]:
print("Data Types:")
print(df.dtypes) # gives the data types of each column

Data Types:
tv           float64
radio        float64
newspaper    float64
sales        float64
dtype: object


In [14]:
print("Number of unique values in each column:")
print(df.nunique()) # gives the number of unique values in each column

Number of unique values in each column:
tv           154
radio        141
newspaper    139
sales        104
dtype: int64


## 7. Data Dictionary

The dataset contains four numerical variables.

| Variable | Description | Unit |
|---|---|---|
| TV | Advertising expenditure on TV | $ thousands |
| Radio | Advertising expenditure on Radio | $ thousands |
| Newspaper | Advertising expenditure on Newspaper | $ thousands |
| Sales | Product sales | Thousand units |

### Understanding the Units

Advertising expenditure is recorded in thousands of dollars.

For example, a TV value of 150 represents an advertising
expenditure of $150,000.

Sales is measured in thousands of units.

For example, a sales value of 20 represents 20,000 units sold.

## 8. Initial Observations

The following observations are based on the training dataset:

- The dataset contains 160 observations and four numerical variables.
- The advertising channels are TV, Radio and Newspaper.
- Sales is the target variable.
- All variables are numerical.
- The dataset is checked for missing values, duplicate records
  and negative values.
- The dataset is suitable for further exploratory data analysis,
  subject to the limitations of the available variables.

Further analysis will investigate the distributions, relationships
and statistical characteristics of these variables.

## 9. Conclusion

The initial data understanding process establishes the structure,
variable definitions and data quality characteristics of the
Advertising dataset.

The training dataset will be used for exploratory data analysis
and model development, while the test dataset is reserved for
final model evaluation.

The next stage focuses on exploratory data analysis to understand
the distributions of advertising expenditure and its relationship
with product sales.